In [2]:
import json
from pathlib import Path
from collections import Counter

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from IPython.display import display, Markdown, HTML

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

PATH = Path(r"data/archive/arxiv-metadata-oai-snapshot.json")
display(Markdown(f"**Файл существует:** `{PATH.exists()}`  \n"
                 f"**Размер:** `{PATH.stat().st_size / 1e9:.2f} GB`"))

**Файл существует:** `True`  
**Размер:** `5.50 GB`

In [3]:
with PATH.open("r", encoding="utf-8") as f:
    first = json.loads(f.readline())

display(Markdown("### Ключи записи"))
display(list(first.keys()))

display(Markdown("### Значения (обрезаны до 200 символов)"))
preview = pd.DataFrame(
    [(k, str(v)[:200]) for k, v in first.items()],
    columns=["field", "value"]
)
display(preview)

### Ключи записи

['id',
 'submitter',
 'authors',
 'title',
 'comments',
 'journal-ref',
 'doi',
 'report-no',
 'categories',
 'license',
 'abstract',
 'versions',
 'update_date',
 'authors_parsed']

### Значения (обрезаны до 200 символов)

,field,value
0,id,0704.0001
1,submitter,Pavel Nadolsky
2,authors,"C. Bal\'azs, E. L. Berger, P. M. Nadolsky, C.-P. Yuan"
3,title,Calculation of prompt diphoton production cross sections at Tevatron and\n LHC energies
4,comments,"37 pages, 15 figures; published version"
5,journal-ref,"Phys.Rev.D76:013009,2007"
6,doi,10.1103/PhysRevD.76.013009
7,report-no,ANL-HEP-PR-07-12
8,categories,hep-ph
9,license,None


In [4]:
N = 50_000

df = pd.read_json(PATH, lines=True, nrows=N)

display(Markdown(f"### Прочитано: `{df.shape[0]:,}` строк, `{df.shape[1]}` колонок"))
display(df.head(3))

### Прочитано: `50,000` строк, `14` колонок

,id,submitter,authors,title,comments,journal-ref,doi,report-no,categories,license,abstract,versions,update_date,authors_parsed
0,704.0001,Pavel Nadolsky,"C. Bal\'azs, E. L. Berger, P. M. Nadolsky, C.-P. Yuan",Calculation of prompt diphoton production cross sections at Tevatron and\n LHC energies,"37 pages, 15 figures; published version","Phys.Rev.D76:013009,2007",10.1103/PhysRevD.76.013009,ANL-HEP-PR-07-12,hep-ph,None,A fully differential calculation in perturbative quantum chromodynamics is\npresented for the production of massiv...,"[{'version': 'v1', 'created': 'Mon, 2 Apr 2007 19:18:42 GMT'}, {'version': 'v2', 'created': 'Tue, 24 Jul 2007 20:10:...",2008-11-26,"[[Balázs, C., ], [Berger, E. L., ], [Nadolsky, P. M., ], [Yuan, C. -P., ]]"
1,704.0002,Louis Theran,Ileana Streinu and Louis Theran,Sparsity-certifying Graph Decompositions,To appear in Graphs and Combinatorics,None,None,None,math.CO cs.CG,http://arxiv.org/licenses/nonexclusive-distrib/1.0/,"We describe a new algorithm, the $(k,\ell)$-pebble game with colors, and use\nit obtain a characterization of the ...","[{'version': 'v1', 'created': 'Sat, 31 Mar 2007 02:26:18 GMT'}, {'version': 'v2', 'created': 'Sat, 13 Dec 2008 17:26...",2008-12-13,"[[Streinu, Ileana, ], [Theran, Louis, ]]"
2,704.0003,Hongjun Pan,Hongjun Pan,The evolution of the Earth-Moon system based on the dark matter field\n fluid model,"23 pages, 3 figures",None,None,None,physics.gen-ph,None,The evolution of Earth-Moon system is described by the dark matter field\nfluid model proposed in the Meeting of D...,"[{'version': 'v1', 'created': 'Sun, 1 Apr 2007 20:46:54 GMT'}, {'version': 'v2', 'created': 'Sat, 8 Dec 2007 23:47:2...",2008-01-13,"[[Pan, Hongjun, ]]"


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 220)

PATH = Path(r"data/archive/arxiv-metadata-oai-snapshot.json")

chunks = []
reader = pd.read_json(
    PATH, lines=True, chunksize=200_000,
    dtype={"id": str},          # критично: иначе float64 и потеря 0704.0001
    convert_dates=False,
)
for chunk in reader:
    chunks.append(chunk)

df = pd.concat(chunks, ignore_index=True)
del chunks

display(Markdown(f"### Загружено: `{df.shape[0]:,}` × `{df.shape[1]}`"))
display(Markdown(f"**id dtype:** `{df['id'].dtype}`"))
display(df["id"].head(5).to_frame())

### Загружено: `3,148,882` × `14`

**id dtype:** `object`

,id
0,0704.0001
1,0704.0002
2,0704.0003
3,0704.0004
4,0704.0005


📦 **JSON → Parquet:** читает arXiv JSON чанками по 200k строк и пишет в `data/parquet/part_*.parquet`, попутно конвертируя `id` в int32 и строки в pyarrow.

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

PATH = Path(r"data/archive/arxiv-metadata-oai-snapshot.json")
OUT  = Path("data/parquet")
OUT.mkdir(parents=True, exist_ok=True)

for f in OUT.glob("part_*.parquet"):
    f.unlink()

reader = pd.read_json(
    PATH, lines=True, chunksize=200_000,
    dtype={"id": str},
    convert_dates=False,
)

log = []
for i, chunk in enumerate(reader):
    id_str = chunk["id"].astype(str)
    has_slash = id_str.str.contains("/", regex=False)

    # готовим три колонки
    id_prefix = pd.Series(pd.NA, index=chunk.index, dtype="string")
    id_ym     = pd.Series(pd.NA, index=chunk.index, dtype="Int32")
    id_num    = pd.Series(pd.NA, index=chunk.index, dtype="Int32")

    # --- старый формат: "acc-phys/9411001" ---
    if has_slash.any():
        old = id_str[has_slash].str.split("/", n=1, expand=True)
        id_prefix.loc[has_slash] = old[0].values
        id_ym.loc[has_slash]     = old[1].str.slice(0, 4).astype("Int32").values
        id_num.loc[has_slash]    = old[1].str.slice(4).astype("Int32").values

    # --- новый формат: "0704.0001" ---
    new_mask = ~has_slash
    if new_mask.any():
        new = id_str[new_mask].str.split(".", n=1, expand=True)
        id_prefix.loc[new_mask] = "new"
        id_ym.loc[new_mask]     = new[0].astype("Int32").values
        id_num.loc[new_mask]    = new[1].astype("Int32").values

    chunk["id_prefix"] = id_prefix.astype("category")
    chunk["id_ym"]     = id_ym
    chunk["id_num"]    = id_num
    chunk = chunk.drop(columns=["id"])

    for col in ["license", "update_date", "categories"]:
        chunk[col] = chunk[col].astype("category")

    for col in ["title", "abstract", "authors", "submitter",
                "comments", "journal-ref", "doi", "report-no"]:
        chunk[col] = chunk[col].astype("string[pyarrow]")

    f = OUT / f"part_{i:04d}.parquet"
    chunk.to_parquet(f, index=False)

    log.append({"part": f.name, "rows": len(chunk)})
    print(f"{f.name}: {len(chunk):,} rows")

display(pd.DataFrame(log))
display(Markdown(f"**Всего частей:** `{len(log)}`  \n"
                 f"**Всего строк:** `{sum(r['rows'] for r in log):,}`"))

part_0000.parquet: 200,000 rows
part_0001.parquet: 200,000 rows
part_0002.parquet: 200,000 rows
part_0003.parquet: 200,000 rows
part_0004.parquet: 200,000 rows
part_0005.parquet: 200,000 rows
part_0006.parquet: 200,000 rows
part_0007.parquet: 200,000 rows
part_0008.parquet: 200,000 rows
part_0009.parquet: 200,000 rows
part_0010.parquet: 200,000 rows
part_0011.parquet: 200,000 rows
part_0012.parquet: 200,000 rows
part_0013.parquet: 200,000 rows
part_0014.parquet: 200,000 rows
part_0015.parquet: 148,882 rows


,part,rows
0,part_0000.parquet,200000
1,part_0001.parquet,200000
2,part_0002.parquet,200000
3,part_0003.parquet,200000
4,part_0004.parquet,200000
5,part_0005.parquet,200000
6,part_0006.parquet,200000
7,part_0007.parquet,200000
8,part_0008.parquet,200000
9,part_0009.parquet,200000


**Всего частей:** `16`  
**Всего строк:** `3,148,882`

In [4]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

SRC = Path("data/parquet")
DST = Path("data/parquet_opt")
DST.mkdir(parents=True, exist_ok=True)

# чистим старые оптимизированные партиции
for f in DST.glob("part_*.parquet"):
    f.unlink()

display(Markdown(f"**Источник:** `{SRC}`  \n**Назначение:** `{DST}`"))

**Источник:** `data\parquet`  
**Назначение:** `data\parquet_opt`

In [5]:
def optimize(df: pd.DataFrame) -> pd.DataFrame:
    """Стандартные приёмы pandas: downcast int, category, string[pyarrow], datetime."""
    df = df.copy()

    # --- 1) Целочисленные: минимально возможный тип ---
    for col in ["id_ym", "id_num"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], downcast="unsigned")  # uint16 / uint32

    # --- 2) Даты: update_date -> datetime64 + производные ---
    if "update_date" in df.columns:
        dt = pd.to_datetime(df["update_date"], errors="coerce")
        df["update_year"]  = dt.dt.year.astype("uint16")
        df["update_month"] = dt.dt.month.astype("uint8")
        df = df.drop(columns=["update_date"])

    # --- 3) Низкокардинальные категории ---
    for col in ["license", "categories", "id_prefix"]:
        if col in df.columns and df[col].dtype.name != "category":
            df[col] = df[col].astype("category")

    # --- 4) Тексты: pyarrow-backed строки ---
    str_cols = ["title", "abstract", "authors", "submitter",
                "comments", "journal-ref", "doi", "report-no"]
    for col in str_cols:
        if col in df.columns:
            df[col] = df[col].astype("string[pyarrow]")

    # --- 5) Списки (versions, authors_parsed) -> сериализуем в string ---
    for col in ["versions", "authors_parsed"]:
        if col in df.columns:
            df[col] = df[col].map(
                lambda v: "" if v is None else str(v),
                na_action="ignore"
            ).astype("string[pyarrow]")

    return df

In [6]:
import pyarrow.parquet as pq

log = []
for src_file in sorted(SRC.glob("part_*.parquet")):
    df = pd.read_parquet(src_file)

    before_mem = df.memory_usage(deep=True).sum() / 1e6
    before_cols = df.shape[1]

    df = optimize(df)

    after_mem = df.memory_usage(deep=True).sum() / 1e6
    dst_file = DST / src_file.name
    df.to_parquet(dst_file, index=False)

    log.append({
        "part":        src_file.name,
        "rows":        len(df),
        "cols_before": before_cols,
        "cols_after":  df.shape[1],
        "mem_before_MB": round(before_mem, 1),
        "mem_after_MB":  round(after_mem, 1),
        "saving_%":      round(100 * (1 - after_mem / before_mem), 1),
        "src_MB":     round(src_file.stat().st_size / 1e6, 1),
        "dst_MB":     round(dst_file.stat().st_size / 1e6, 1),
    })
    print(f"{src_file.name}: {before_mem:.1f} -> {after_mem:.1f} MB")

log_df = pd.DataFrame(log)
display(Markdown("### Результат по партициям"))
display(log_df)
display(Markdown(f"**Итого RAM:** `{log_df.mem_before_MB.sum()/1000:.2f} GB -> "
                 f"{log_df.mem_after_MB.sum()/1000:.2f} GB`"))
display(Markdown(f"**Итого диск:** `{log_df.src_MB.sum()/1000:.2f} GB -> "
                 f"{log_df.dst_MB.sum()/1000:.2f} GB`"))

part_0000.parquet: 343.4 -> 282.4 MB
part_0001.parquet: 355.7 -> 303.7 MB
part_0002.parquet: 364.3 -> 316.5 MB
part_0003.parquet: 371.4 -> 323.5 MB
part_0004.parquet: 378.8 -> 330.5 MB
part_0005.parquet: 385.1 -> 337.0 MB
part_0006.parquet: 393.7 -> 347.1 MB
part_0007.parquet: 399.5 -> 352.6 MB
part_0008.parquet: 402.2 -> 356.1 MB
part_0009.parquet: 408.3 -> 362.6 MB
part_0010.parquet: 414.2 -> 369.7 MB
part_0011.parquet: 420.1 -> 377.8 MB
part_0012.parquet: 426.7 -> 382.3 MB
part_0013.parquet: 426.2 -> 373.6 MB
part_0014.parquet: 325.7 -> 259.7 MB
part_0015.parquet: 225.4 -> 172.6 MB


### Результат по партициям

,part,rows,cols_before,cols_after,mem_before_MB,mem_after_MB,saving_%,src_MB,dst_MB
0,part_0000.parquet,200000,16,17,343.4,282.4,17.8,142.0,144.0
1,part_0001.parquet,200000,16,17,355.7,303.7,14.6,152.0,153.9
2,part_0002.parquet,200000,16,17,364.3,316.5,13.1,158.6,160.4
3,part_0003.parquet,200000,16,17,371.4,323.5,12.9,163.7,165.2
4,part_0004.parquet,200000,16,17,378.8,330.5,12.7,168.6,169.8
5,part_0005.parquet,200000,16,17,385.1,337.0,12.5,173.1,174.1
6,part_0006.parquet,200000,16,17,393.7,347.1,11.9,178.5,179.6
7,part_0007.parquet,200000,16,17,399.5,352.6,11.7,182.3,182.8
8,part_0008.parquet,200000,16,17,402.2,356.1,11.5,183.7,184.5
9,part_0009.parquet,200000,16,17,408.3,362.6,11.2,187.4,187.9


**Итого RAM:** `6.04 GB -> 5.25 GB`

**Итого диск:** `2.66 GB -> 2.68 GB`

In [10]:
from pathlib import Path
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display, Markdown

# ---------- пути ----------
SRC     = Path("data/parquet")
OUT_ROOT = Path("data/optimized_v1")   # общая папка версии
META  = OUT_ROOT / "meta"
TEXT  = OUT_ROOT / "texts"
AUTH  = OUT_ROOT / "authors"
VERS  = OUT_ROOT / "versions"

for d in (META, TEXT, AUTH, VERS):
    d.mkdir(parents=True, exist_ok=True)
    for f in d.glob("part_*.parquet"):
        f.unlink()

ID_COLS   = ["id_prefix", "id_ym", "id_num"]
META_COLS = ID_COLS + ["categories", "license", "update_date",
                       "submitter", "comments", "journal-ref",
                       "doi", "report-no", "n_versions", "n_authors"]
TEXT_COLS = ID_COLS + ["title", "abstract"]

SEQ_TYPES = (list, tuple, np.ndarray)


def write_parquet(df: pd.DataFrame, path: Path):
    """DataFrame -> Parquet с zstd и dictionary encoding для категориальных."""
    t = pa.Table.from_pandas(df, preserve_index=False)
    for i, field in enumerate(t.schema):
        if pa.types.is_dictionary(field.type):
            continue
        if field.name in df.columns and isinstance(df[field.name].dtype, pd.CategoricalDtype):
            t = t.set_column(i, field.name, t.column(i).dictionary_encode())
    pq.write_table(t, path, compression="zstd", compression_level=10,
                   use_dictionary=True)


log = []
for src_file in sorted(SRC.glob("part_*.parquet")):
    df = pd.read_parquet(src_file)
    rows = len(df)

    # ---------- id ----------
    df["id_ym"]     = df["id_ym"].astype("uint16")
    df["id_num"]    = df["id_num"].astype("uint32")
    df["id_prefix"] = df["id_prefix"].astype("category")

    # ---------- update_date ----------
    df["update_date"] = pd.to_datetime(df["update_date"], errors="coerce")

    # ---------- versions ----------
    df["n_versions"] = df["versions"].map(
        lambda v: len(v) if isinstance(v, SEQ_TYPES) else 0
    ).astype("uint8")

    v_rows = []
    for ids, v in zip(df[ID_COLS].to_dict("records"), df["versions"].tolist()):
        if isinstance(v, SEQ_TYPES):
            for x in v:
                if isinstance(x, dict):
                    v_rows.append({
                        **ids,
                        "version": x.get("version"),
                        "created": pd.to_datetime(x.get("created"), errors="coerce"),
                    })
    if v_rows:
        v_df = pd.DataFrame(v_rows)
        v_df["version"] = v_df["version"].astype("category")
        write_parquet(v_df, VERS / src_file.name)

    df = df.drop(columns=["versions"])

    # ---------- authors_parsed ----------
    df["n_authors"] = df["authors_parsed"].map(
        lambda a: len(a) if isinstance(a, SEQ_TYPES) else 0
    ).astype("uint8")

    a_rows = []
    for ids, auths in zip(df[ID_COLS].to_dict("records"), df["authors_parsed"].tolist()):
        if isinstance(auths, SEQ_TYPES):
            for order, a in enumerate(auths):
                if isinstance(a, SEQ_TYPES) and len(a) >= 2:
                    a_rows.append({
                        **ids,
                        "author_order": order,
                        "last":   a[0] or None,
                        "first":  a[1] or None,
                        "suffix": (a[2] if len(a) > 2 else "") or None,
                    })
    if a_rows:
        a_df = pd.DataFrame(a_rows)
        a_df["author_order"] = a_df["author_order"].astype("uint8")
        for c in ["last", "first", "suffix"]:
            a_df[c] = a_df[c].astype("string[pyarrow]")
        write_parquet(a_df, AUTH / src_file.name)

    df = df.drop(columns=["authors_parsed"])

    # ---------- категории и строки ----------
    for c in ["categories", "license"]:
        df[c] = df[c].astype("category")
    for c in ["submitter", "comments", "journal-ref", "doi", "report-no"]:
        df[c] = df[c].astype("string[pyarrow]")

    # ---------- split meta / texts ----------
    meta  = df[META_COLS].copy()
    texts = df[TEXT_COLS].copy()
    texts["title"]    = texts["title"].astype("string[pyarrow]")
    texts["abstract"] = texts["abstract"].astype("string[pyarrow]")

    write_parquet(meta,  META / src_file.name)
    write_parquet(texts, TEXT / src_file.name)

    log.append({
        "part":     src_file.name,
        "rows":     rows,
        "meta_MB":  (META / src_file.name).stat().st_size / 1e6,
        "texts_MB": (TEXT / src_file.name).stat().st_size / 1e6,
        "auth_MB":  (AUTH / src_file.name).stat().st_size / 1e6 if a_rows else 0,
        "vers_MB":  (VERS / src_file.name).stat().st_size / 1e6 if v_rows else 0,
        "authors":  len(a_rows),
        "versions": len(v_rows),
    })
    print(f"{src_file.name}: meta={log[-1]['meta_MB']:.1f} "
          f"texts={log[-1]['texts_MB']:.1f} "
          f"auth={log[-1]['auth_MB']:.1f} "
          f"vers={log[-1]['vers_MB']:.1f} MB "
          f"({log[-1]['authors']:,} authors, {log[-1]['versions']:,} versions)")

log_df = pd.DataFrame(log)
display(Markdown("### Размеры по таблицам"))
display(log_df)

totals = log_df[["meta_MB", "texts_MB", "auth_MB", "vers_MB"]].sum()
display(Markdown(
    f"**meta:**     `{totals['meta_MB']/1000:.2f} GB`  \n"
    f"**texts:**    `{totals['texts_MB']/1000:.2f} GB`  \n"
    f"**authors:**  `{totals['auth_MB']/1000:.2f} GB` ({log_df['authors'].sum():,} строк)  \n"
    f"**versions:** `{totals['vers_MB']/1000:.2f} GB` ({log_df['versions'].sum():,} строк)  \n"
    f"**Итого:**    `{totals.sum()/1000:.2f} GB`"
))

part_0000.parquet: meta=7.0 texts=56.8 auth=3.7 vers=2.4 MB (677,081 authors, 316,988 versions)
part_0001.parquet: meta=6.9 texts=60.0 auth=4.6 vers=2.4 MB (865,652 authors, 323,416 versions)
part_0002.parquet: meta=6.8 texts=62.5 auth=5.2 vers=2.4 MB (966,246 authors, 328,521 versions)
part_0003.parquet: meta=6.6 texts=65.1 auth=5.5 vers=2.4 MB (967,383 authors, 333,599 versions)
part_0004.parquet: meta=6.5 texts=67.5 auth=5.8 vers=2.5 MB (974,999 authors, 335,653 versions)
part_0005.parquet: meta=6.4 texts=69.9 auth=6.3 vers=2.5 MB (987,383 authors, 336,017 versions)
part_0006.parquet: meta=6.3 texts=73.6 auth=6.9 vers=2.6 MB (998,629 authors, 338,837 versions)
part_0007.parquet: meta=6.2 texts=75.4 auth=7.1 vers=2.5 MB (1,058,676 authors, 334,354 versions)
part_0008.parquet: meta=6.0 texts=76.3 auth=7.3 vers=2.5 MB (1,050,639 authors, 333,609 versions)
part_0009.parquet: meta=5.8 texts=77.9 auth=7.5 vers=2.5 MB (1,081,604 authors, 333,960 versions)
part_0010.parquet: meta=5.6 texts=

### Размеры по таблицам

,part,rows,meta_MB,texts_MB,auth_MB,vers_MB,authors,versions
0,part_0000.parquet,200000,7.035938,56.815810,3.725676,2.354163,677081,316988
1,part_0001.parquet,200000,6.919881,60.029975,4.626693,2.356036,865652,323416
2,part_0002.parquet,200000,6.800972,62.537274,5.244460,2.395598,966246,328521
3,part_0003.parquet,200000,6.610104,65.072806,5.501875,2.433391,967383,333599
4,part_0004.parquet,200000,6.459913,67.501584,5.790018,2.470365,974999,335653
5,part_0005.parquet,200000,6.377401,69.889091,6.297705,2.480993,987383,336017
6,part_0006.parquet,200000,6.326600,73.583761,6.861882,2.578632,998629,338837
7,part_0007.parquet,200000,6.219078,75.370096,7.136448,2.544541,1058676,334354
8,part_0008.parquet,200000,6.039500,76.274071,7.294937,2.536310,1050639,333609
9,part_0009.parquet,200000,5.800435,77.933606,7.450168,2.540102,1081604,333960


**meta:**     `0.10 GB`  
**texts:**    `1.09 GB`  
**authors:**  `0.09 GB` (14,989,782 строк)  
**versions:** `0.04 GB` (5,030,734 строк)  
**Итого:**    `1.32 GB`